# How to train a GNS using the Pipeline


## Import classes and define paths


In [ ]:
from pyLOM import NN
from pyLOM.NN.utils.config_schema import GNSModelConfig, GNSTrainingConfig
from pyLOM.NN.utils.experiment import plot_train_test_loss, plot_true_vs_pred, save_experiment_artifacts
from pyLOM.utils.config_resolvers import load_yaml
from pathlib import Path
from dacite import from_dict, Config as DaciteConfig
from dataclasses import asdict
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [ ]:
CONFIG_PATH = Path.cwd() / 'GNS_training_config.yaml'
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path.cwd() / 'notebook_examples/NN/GNS_training_config.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Could not find config file. Tried: {Path.cwd() / "GNS_training_config.yaml"} and {Path.cwd() / "notebook_examples/NN/GNS_training_config.yaml"}')
cfg = load_yaml(CONFIG_PATH)
def _resolve(path_like: str) -> Path:
    p = Path(path_like)
    return p if p.is_absolute() else (CONFIG_PATH.parent / p).resolve()
TRAIN_PATH = _resolve(cfg['datasets']['train_ds'])
TEST_PATH = _resolve(cfg['datasets']['test_ds'])
GRAPH_PATH = _resolve(cfg['model']['graph_path'])
CASE_DIR = _resolve(cfg['experiment']['results_path'])
for path in (TRAIN_PATH, TEST_PATH, GRAPH_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Missing required input file: {path}')
NN.create_results_folder(CASE_DIR)
NN.create_results_folder(CASE_DIR / 'models')
NN.create_results_folder(CASE_DIR / 'plots')
print('Config:', CONFIG_PATH)
print('Using files:')
print('  train:', TRAIN_PATH)
print('  test :', TEST_PATH)
print('  graph:', GRAPH_PATH)
print('Results:', CASE_DIR)


## Create datasets
For this GNS example, we use the cylinder train/test files and predict the field `VELOX`.


In [ ]:
dataset_cfg = cfg['dataset_config']
ds_kwargs = {
    'field_names': dataset_cfg['field_names'],
    'variables_names': dataset_cfg.get('variables_names', ['all']),
    'add_variables': bool(dataset_cfg.get('add_variables', True)),
    'add_mesh_coordinates': bool(dataset_cfg.get('add_mesh_coordinates', False)),
    'squeeze_last_dim': False,
    'channels_last': True,
}
if dataset_cfg.get('mesh_shape') is not None:
    ds_kwargs['mesh_shape'] = tuple(dataset_cfg['mesh_shape'])
train_dataset = NN.Dataset.load(TRAIN_PATH, **ds_kwargs)
test_dataset = NN.Dataset.load(TEST_PATH, **ds_kwargs)


In [ ]:
x_train, y_train = train_dataset[:]
x_test, y_test = test_dataset[:]
print('Train dataset length:', len(train_dataset))
print('Test dataset length :', len(test_dataset))
print('X_train shape:', x_train.shape, '| y_train shape:', y_train.shape)
print('X_test  shape:', x_test.shape,  '| y_test  shape:', y_test.shape)


## Model creation
GNS uses the same config-driven workflow style as MLP/KAN.


In [ ]:
dcfg = DaciteConfig(strict=True)
model_cfg = from_dict(GNSModelConfig, cfg['model']['config'], config=dcfg)
training_cfg = from_dict(GNSTrainingConfig, cfg['training'], config=dcfg)


In [ ]:
model = NN.GNS.from_graph_path(config=model_cfg, graph_path=GRAPH_PATH)
print(model)


## Run the pipeline
Here we use `test_dataset` also as validation dataset to obtain `test_loss` during training.


In [ ]:
pipeline = NN.Pipeline(
    train_dataset=train_dataset,
    valid_dataset=test_dataset,
    test_dataset=test_dataset,
    model=model,
    training_params={'config': training_cfg},
)
training_logs = pipeline.run()


## Show plots


In [ ]:
loss_dict = {'GNS': {'train': training_logs.get('train_loss', []), 'val': training_logs.get('test_loss', [])}}
plot_train_test_loss(loss_dict, show=True, save=False, yscale='log')


## Evaluate the model with some metrics


In [ ]:
evaluator = NN.RegressionEvaluator()
metrics = evaluator(y_true_flat, preds_flat)
evaluator.print_metrics()


## Save experiment artifacts\n

In [ ]:
artifacts_dir = save_experiment_artifacts(
    base_path=CASE_DIR,
    model=pipeline.model,
    metrics_dict={k: float(v) for k, v in metrics.items()},
    full_run_config={
        'experiment': cfg.get('experiment', {}),
        'datasets': cfg.get('datasets', {}),
        'dataset_config': cfg.get('dataset_config', {}),
        'model': {'graph_path': str(GRAPH_PATH), 'config': asdict(model_cfg)},
        'training': asdict(training_cfg),
    },
    extra_files={
        'train_val_loss.png': lambda p: plot_train_test_loss(
            {'GNS': {'train': training_logs.get('train_loss', []), 'val': training_logs.get('test_loss', [])}},
            save=True,
            save_path=str(p),
            show=False,
            yscale='log',
        ),
        'true_vs_pred.png': lambda p: plot_true_vs_pred(y_true_flat, preds_flat, p),
    },
    return_path=True,
)
print('Artifacts saved in:', artifacts_dir)


As you can see, this example follows the same Pipeline workflow as the MLP and KAN notebooks. The main differences are graph-based model construction (`GNS.from_graph_path`) and graph data handling.


## Export predictions to ParaView format for visualization

In [ ]:
from pyLOM import Mesh
from pyLOM.NN.utils.experiment import ParaviewExportConfig, export_predictions_to_paraview

mesh = Mesh.load(str(TEST_PATH), mpio=False)
nsnaps = int(preds.shape[0])
field_names = cfg['dataset_config']['field_names']
if not field_names:
    raise ValueError('dataset_config.field_names must contain at least one field')
field_name = str(field_names[0])

# Build per-snapshot input metadata and inverse-scale it when a scaler is available.
x_meta = x_test.detach().cpu().numpy()
x_meta = test_dataset.inverse_scale_inputs(x_meta)
if x_meta.ndim == 1:
    x_meta = x_meta.reshape(-1, 1)

input_names = list(cfg['dataset_config'].get('variables_names', []))
if len(input_names) != x_meta.shape[1]:
    input_names = [f'input_{i}' for i in range(x_meta.shape[1])]
snapshot_metadata = {str(name): x_meta[:, i] for i, name in enumerate(input_names)}

pv_cfg = ParaviewExportConfig(
    mesh=mesh,
    cell_order=mesh.cellOrder,
    partition_table=mesh.partition_table,
    instants=np.arange(nsnaps, dtype=np.int32),
    times=np.arange(nsnaps, dtype=np.float64),
    output_dir=Path(artifacts_dir) / 'paraview',
    base_name='gns_predictions',
    mode='single',
    snapshot_metadata=snapshot_metadata,
    extra_cell_fields={field_name: y_true},
)

written_files = export_predictions_to_paraview(
    y_pred=preds,
    y_true=y_true,
    metrics=[
        (f'{field_name}_pred', lambda yp, yt: yp),
        ('error', lambda yp, yt: np.abs(yp - yt)),
    ],
    config=pv_cfg,
)

print('ParaView files written:')
for wf in written_files:
    print(' -', wf)
